#### Classical Machine Learning for Text

- **Goal:** Establish a strong baseline. Understand how to turn text into numbers and the mechanics of classification (Weights & Gradients).
  - **Day 1 (Vectorization):** Loading **IMDB**. Use TF-IDF to create text features; Converting Sparse Matrices to PyTorch Tensors.
 
- **Sources:**
  - text-feature-extraction: https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
  - huggingface datasets: https://huggingface.co/datasets/stanfordnlp/imdb

In [ ]:
# suppress warnings 
from warnings import filterwarnings
filterwarnings("ignore") 

# import the appropriate libraries
import pandas as pd
import torch
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer #text feature extraction 
from utils import get_imdb_ds, create_dataloaders, save_tensors, load_tensors_and_create_dataloaders

# load in environment such HF_token
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Load and prepare dataset using utility function (now includes validation split)
train_df, val_df, test_df = get_imdb_ds(seed=204, train_size=1000, val_size=200, test_size=250)

In [3]:
# Display first few rows
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

In [4]:
train_df.head()

,text,label
0,"When I saw this movie, I couldn't believe my e...",0
1,LOL!!! delirious was so funny.. i was in tears...,1
2,This movie was bad from the start. The only pu...,0
3,Richard Attenborough who already given us magn...,1
4,This ranks as one of the worst movies I've see...,0


#### **Dataset Breakdown**
- **Feature:** Free form text
- **Label:** Binary, 0 is negative and 1 is positive. (Note: For multi-class classification, e.g., 20 topics, labels must be numeric integers from 0 to 19).

Machine Learning models require numerical features of fixed size. Vectorization is a method of converting text of variable length into numerical features of fixed size.

### **Vectorization**
The general process of turning a collection of text documents into numerical feature vectors. This is used for text feature extraction. 

#### **Vectorization Steps**
- **Tokenization:** Break text into separate tokens (e.g., using white space as a separator). Each unique token is mapped to a specific column index (integer ID) in the feature matrix.
- **Counting:** Count occurrences of each token in a given text/document to create a frequency vector.
- **Weighting & Normalization (Optional/Method Dependent):** 
  - **Weighting (IDF):** Reduce the weight of tokens that appear frequently across the entire corpus (like "the", "is") as they are less informative.
  - **Normalization ($L_2$):** Scale the final vector so that it has a unit norm (length of 1), allowing for comparison between documents of different lengths.

#### **Popular Vectorization Concepts**
- **Bag of Words (or Bag of n-grams):** Uses **raw counts** of tokens. It generally has **no weighting** (all words are treated equally based on frequency) and **no normalization** (longer documents result in vectors with larger magnitudes).
  - **Corpus:** Represented by a matrix with one row per document and one column per token occurring in the corpus.
- **Sparsity:** Most documents only contain a very small subset of the entire corpus vocabulary. Thus, the matrix for the corpus will be very sparse (mostly zeros). The `scipy.sparse` package is used to store the matrix in a sparse representation.
- **Using Stop Words:** It can be dangerous to remove stop words as they can sometimes be helpful for feature understanding (e.g., "not"). They are not always uninformative. It is also important to ensure the stop word list you add uses the same preprocessing and tokenization as the vectorizer class you use.
- **TF-IDF Term Weighting:** Bag of words doesn't account for document frequency. TF-IDF adds both **Weighting** (IDF) and **Normalization** ($L_2$). 
  
  $$ \text{tf-idf}(t,d) = \text{tf}(t,d) \times \text{idf}(t) $$

   - **Term Frequency (tf):** Number of times a term occurs in a given document.
   - **Inverse Document Frequency (idf):** 
     
     $$ \text{idf}(t) = \log\left(\frac{1+n}{1+\text{df}(t)}\right) + 1 $$
     
     Where $n$ is the number of documents in the document set and $\text{df}(t)$ is the number of documents in the document set that contain the term $t$. 
     
     TF-IDF vectors are often normalized by the Euclidean norm:
     
     $$ v_{norm} = \frac{v}{||v||_2} $$

### **Mini-Example: Bag of Words vs. TF-IDF**
Before applying this to the full IMDB dataset, let's look at a toy example to see the difference between raw counts (Bag of Words) and weighted scores (TF-IDF).

Notice how in TF-IDF:
1. Common words like "this" and "movie" (which appear in all 3 docs) have **lower scores** because they are less unique.
2. Unique words like "love", "hate", and "okay" have **higher scores**.

In [ ]:
# Fake Corpus
corpus = [
    "I love this movie",
    "I hate this movie",
    "This movie is okay"
]

# 1. Bag of Words (Raw Counts)
# Note: By default, single characters like 'I' are ignored (token_pattern='(?u)\b\w\w+\b')
count_vec = CountVectorizer()
bow_matrix = count_vec.fit_transform(corpus)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=count_vec.get_feature_names_out(), index=["Doc 1", "Doc 2", "Doc 3"])

# 2. TF-IDF (Weighted Scores)
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(corpus)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vec.get_feature_names_out(), index=["Doc 1", "Doc 2", "Doc 3"])

print("--- Bag of Words (Raw Counts) ---")
print(bow_df)
print("\n--- TF-IDF (Weighted Scores) ---")
print(tfidf_df.round(2))

--- Bag of Words (Raw Counts) ---
       hate  is  love  movie  okay  this
Doc 1     0   0     1      1     0     1
Doc 2     1   0     0      1     0     1
Doc 3     0   1     0      1     1     1

--- TF-IDF (Weighted Scores) ---
       hate    is  love  movie  okay  this
Doc 1  0.00  0.00  0.77   0.45  0.00  0.45
Doc 2  0.77  0.00  0.00   0.45  0.00  0.45
Doc 3  0.00  0.61  0.00   0.36  0.61  0.36


In [81]:
X_train_text = train_df["text"]
y_train = train_df["label"]

X_val_text = val_df["text"]
y_val = val_df["label"]

X_test_text = test_df["text"]
y_test = test_df["label"]
max_features = 2000 # Selects the top 2000 most frequent words (after stop words removal)

## TF-IDF

In [90]:
# min_df=5: Ignore terms that appear in less than 5 documents (removes noise/typos)
# max_df=0.8: Ignore terms that appear in more than 80% of documents (removes corpus-specific stop words)
tfidf_vectorizer = TfidfVectorizer(max_features=max_features, stop_words="english", min_df=5, max_df=0.8) 
X_train_tfidf_sparse = tfidf_vectorizer.fit_transform(X_train_text)
X_val_tfidf_sparse = tfidf_vectorizer.transform(X_val_text)
X_test_tfidf_sparse = tfidf_vectorizer.transform(X_test_text)

## Convert to PyTorch Tensor for Dense Array

In [91]:
X_train_dense = X_train_tfidf_sparse.toarray()
X_val_dense = X_val_tfidf_sparse.toarray()
X_test_dense = X_test_tfidf_sparse.toarray()
X_train_tensor = torch.tensor(X_train_dense, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_dense, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_dense, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

In [93]:
print("\n--- Ready for Day 2 (Model Building) ---")
print(f"X_train_tensor: {X_train_tensor.shape} (Type: {X_train_tensor.dtype})")
print(f"y_train_tensor: {y_train_tensor.shape} (Type: {y_train_tensor.dtype})")
print(f"X_val_tensor: {X_val_tensor.shape} (Type: {X_val_tensor.dtype})")
print(f"y_val_tensor: {y_val_tensor.shape} (Type: {y_val_tensor.dtype})")
print(f"X_test_tensor: {X_test_tensor.shape} (Type: {X_test_tensor.dtype})")
print(f"y_test_tensor: {y_test_tensor.shape} (Type: {y_test_tensor.dtype})")


--- Ready for Day 2 (Model Building) ---
X_train_tensor: torch.Size([1000, 18166]) (Type: torch.float32)
y_train_tensor: torch.Size([1000, 1]) (Type: torch.float32)
X_train_tensor: torch.Size([250, 18166]) (Type: torch.float32)
y_train_tensor: torch.Size([250, 1]) (Type: torch.float32)


In [96]:
# To see the vocabulary:
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\nRandom 10 words in vocabulary: {feature_names[1050:1060]}")


Random 10 words in vocabulary: ['arrogance' 'arrogant' 'arrow' 'arsenical' 'art' 'artful' 'artfully'
 'arthritic' 'arthur' 'article']


## Create PyTorch DataLoaders and Save

We will wrap the tensors into a `TensorDataset` and create a `DataLoader` for batching. We also save the processed tensors to disk so they can be loaded directly in the next session (Day 2).

In [ ]:
# Create DataLoaders using utility
train_loader, val_loader, test_loader = create_dataloaders(X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor, batch_size=32)

# Save tensors for Day 2 using utility
save_tensors(X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor, 'imdb_tensors.pt')

## How to Load Data on Day 2

Here is the code snippet you would use at the start of your Day 2 notebook to load the processed tensors and recreate the DataLoaders.

In [ ]:
# Load data and recreate DataLoaders using utility
train_loader_loaded, val_loader_loaded, test_loader_loaded = load_tensors_and_create_dataloaders('imdb_tensors.pt', batch_size=32)

print("Day 2 Data Loaded Successfully!")
print(f"Train batch count: {len(train_loader_loaded)}")

## Bonus: Scikit-Learn Logistic Regression with Sparse Features

If you are not using PyTorch/Neural Networks, you can use the sparse TF-IDF matrices (`scipy.sparse.csr_matrix`) **directly** as features in Scikit-Learn models. You do not need to convert them to dense arrays. This is significantly more memory efficient and faster for high-dimensional text data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Initialize Logistic Regression
# Note: We use the sparse matrices (X_train_tfidf_sparse) directly, not the dense tensors
clf = LogisticRegression(random_state=42, max_iter=1000)

# Fit the model
clf.fit(X_train_tfidf_sparse, y_train)

# Predict on test set
y_pred = clf.predict(X_test_tfidf_sparse)

# Evaluate
acc = accuracy_score(y_test, y_pred)
print(f"Sklearn Logistic Regression Accuracy: {acc:.4f}")